# Day 14 / 42: Logistic Regression
### 42 Days of ML Challenge | @VaishnaviJagtap18

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/VaishnaviJagtap18/42-days-aiml-challenge/blob/main/week2_feature_work/day14_logistic_regression/day14_notebook.ipynb)

---

## What You Will Learn
- Why linear regression breaks for classification (with proof — predictions outside 0 and 1)
- The sigmoid function: how it squashes any number into a valid probability
- Decision boundaries and how the threshold (default 0.5) controls predictions
- Build a survival classifier on a Titanic-style dataset, with full evaluation
- The accuracy paradox: a 94.5% accurate model that catches ZERO fraud cases

---

## Step 0: Install and Import

In [ ]:
!pip install numpy pandas matplotlib scikit-learn --quiet

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, confusion_matrix, classification_report, roc_auc_score
)
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)
print("All imports successful. You are ready for Day 14.")

---
## Step 1: Why Linear Regression Fails for Classification

Classification means predicting a category: survived or not, fraud or not, spam or not.

What if you just used linear regression and treated 0/1 as numbers?

`ŷ = w·x + b` can output ANY number: -5, 0.3, 1.8, 100.

But a probability must be between 0 and 1. A prediction of "1.8" or "-0.2" has no meaning.

Let's prove this breaks, with real numbers.

In [ ]:
# Simple example: x < 5 -> class 0, x >= 5 -> class 1
np.random.seed(42)
n = 100
x = np.random.uniform(0, 10, n)
y_binary = (x > 5).astype(int)

# Fit LINEAR regression on a 0/1 target
lin_model = LinearRegression().fit(x.reshape(-1,1), y_binary)
lin_preds = lin_model.predict(x.reshape(-1,1))

print("=== Linear Regression treating 0/1 as numbers ===")
print(f"Prediction range: {lin_preds.min():.4f} to {lin_preds.max():.4f}")
print(f"Predictions below 0: {(lin_preds < 0).sum()} out of {n}")
print(f"Predictions above 1: {(lin_preds > 1).sum()} out of {n}")
print()
print("A 'probability' of -0.05 or 1.12 is meaningless.")
print("Linear regression has no way to keep outputs inside a valid range.")
print()

# Now add ONE outlier far from the rest
x_outlier = np.append(x, 50.0)
y_outlier = np.append(y_binary, 1)

lin_out = LinearRegression().fit(x_outlier.reshape(-1,1), y_outlier)
log_out = LogisticRegression().fit(x_outlier.reshape(-1,1), y_outlier)

test_points = np.array([[2.0], [5.0], [8.0]])
print("=== Effect of ONE outlier (x=50) on predictions for x=2, 5, 8 ===")
print(f"{'x':>6} | {'Linear pred':>14} | {'Logistic prob':>15}")
print("-" * 42)
for tp, lp, lgp in zip(test_points.flatten(), lin_out.predict(test_points), log_out.predict_proba(test_points)[:,1]):
    print(f"{tp:>6.1f} | {lp:>14.4f} | {lgp:>15.4f}")

print()
print("The linear model's predictions shift toward 0.3-0.6 — dragged by the outlier.")
print("The logistic model stays sensible: low prob at x=2, ~0.5 at x=5, high at x=8.")
print("Logistic regression is far less sensitive to outliers in the input.")

---
## Step 2: The Sigmoid Function — The Fix

Logistic regression solves this with one extra step. Instead of using `w·x + b` directly as the prediction, it passes that number through the **sigmoid function**:

$$\sigma(z) = \frac{1}{1 + e^{-z}}$$

No matter what number `z` is (even -1000 or +1000), sigmoid squashes it into the range (0, 1).

- Very negative z → sigmoid close to 0
- z = 0 → sigmoid = exactly 0.5
- Very positive z → sigmoid close to 1

This output IS a valid probability.

In [ ]:
def sigmoid(z):
    return 1 / (1 + np.exp(-z))

test_z = np.array([-10, -5, -1, 0, 1, 5, 10], dtype=float)

print(f"{'z (= w·x + b)':>15} | {'sigmoid(z)':>12}")
print("-" * 32)
for z in test_z:
    print(f"{z:>15.1f} | {sigmoid(z):>12.4f}")

print()
print("Any input -> output always between 0 and 1. This IS a valid probability.")

# Plot the sigmoid curve
z_range = np.linspace(-10, 10, 200)
plt.figure(figsize=(8, 5))
plt.plot(z_range, sigmoid(z_range), color='steelblue', linewidth=2.5)
plt.axhline(0.5, color='red', linestyle='--', label='Decision boundary (threshold=0.5)')
plt.axvline(0, color='gray', linestyle=':', alpha=0.6)
plt.xlabel('z = w·x + b')
plt.ylabel('sigmoid(z) = probability')
plt.title('The Sigmoid Function')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('day14_sigmoid.png', dpi=120, bbox_inches='tight')
plt.show()

print()
print("Decision rule: if sigmoid(z) >= 0.5, predict class 1. Otherwise predict class 0.")
print("sigmoid(z) = 0.5 happens exactly when z = 0, i.e. when w·x + b = 0.")
print("That equation (w·x + b = 0) IS the decision boundary.")

---
## Step 3: Build a Survival Classifier (Titanic-Style)

Predict survival from: age, fare paid, ticket class (1=best, 3=worst), and sex.

This mirrors the real Titanic dataset's structure: women, higher class, and younger passengers had higher survival rates.

In [ ]:
np.random.seed(42)
n = 400

age    = np.random.randint(1, 75, n).astype(float)
fare   = np.random.exponential(30, n) + 5
pclass = np.random.choice([1, 2, 3], n, p=[0.25, 0.25, 0.5]).astype(float)
sex    = np.random.choice([0, 1], n)   # 0 = male, 1 = female

# Build a true survival probability, then sample actual outcomes
logit = -1 + 2.5*sex + 1.2*(3 - pclass) - 0.02*age + 0.01*fare
true_prob = sigmoid(logit)
survived = (np.random.rand(n) < true_prob).astype(int)

df = pd.DataFrame({
    'age': age, 'fare': fare, 'pclass': pclass, 'sex': sex, 'survived': survived
})

print("Sample data (sex: 0=male, 1=female):")
print(df.head(8).round(2).to_string(index=False))
print(f"\nDataset size: {n}")
print(f"Survival rate: {survived.mean()*100:.1f}%")

In [ ]:
X = df[['age', 'fare', 'pclass', 'sex']].values
y = df['survived'].values

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)

y_pred  = model.predict(X_test)
y_proba = model.predict_proba(X_test)[:, 1]

feature_names = ['age', 'fare', 'pclass', 'sex']
print("=== Coefficients (in z = w·x + b, before sigmoid) ===")
for name, coef in zip(feature_names, model.coef_[0]):
    direction = "increases" if coef > 0 else "decreases"
    print(f"  {name:>8}: {coef:>8.4f}   -> {direction} survival log-odds")

print()
print("Reading this: 'sex' has the largest positive coefficient (sex=1 is female).")
print("'pclass' has a negative coefficient: class 3 (worst) lowers survival odds.")
print("This matches the historical pattern: women and first-class passengers")
print("had higher survival rates.")

---
## Step 4: Evaluation — Accuracy, Confusion Matrix, ROC-AUC

In [ ]:
acc = accuracy_score(y_test, y_pred)
auc = roc_auc_score(y_test, y_proba)
cm  = confusion_matrix(y_test, y_pred)

print(f"Accuracy: {acc:.4f}")
print(f"ROC-AUC:  {auc:.4f}")
print()
print("Confusion Matrix:")
print(f"                 Predicted: Died   Predicted: Survived")
print(f"Actual: Died          {cm[0,0]:>4}              {cm[0,1]:>4}")
print(f"Actual: Survived      {cm[1,0]:>4}              {cm[1,1]:>4}")
print()
print(classification_report(y_test, y_pred, target_names=['Died','Survived']))

# Show a few example predictions with probabilities
print("=== Sample predictions with probabilities ===")
print(f"{'Age':>5} {'Fare':>7} {'Class':>6} {'Sex':>5} | {'Actual':>7} {'Pred':>5} {'Prob':>6}")
print("-" * 50)
for i in range(8):
    a,f,p,s = X_test[i]
    sex_label = 'F' if s==1 else 'M'
    print(f"{a:>5.0f} {f:>7.1f} {p:>6.0f} {sex_label:>5} | {y_test[i]:>7} {y_pred[i]:>5} {y_proba[i]:>6.3f}")

---
## Step 5: The Decision Boundary — Threshold Controls Everything

Logistic regression outputs a probability. The **threshold** (default 0.5) decides where probability becomes class 0 vs class 1.

Changing the threshold changes your predictions WITHOUT retraining the model.

This matters enormously when classes are imbalanced — which is the next section.

In [ ]:
print("=== Effect of changing the threshold (same model, same predictions) ===")
print(f"{'Threshold':>10} | {'Predicted Survived':>20} | {'Predicted Died':>16}")
print("-" * 52)
for thresh in [0.3, 0.5, 0.7]:
    preds_t = (y_proba >= thresh).astype(int)
    print(f"{thresh:>10.1f} | {preds_t.sum():>20} | {(preds_t==0).sum():>16}")

print()
print("Lower threshold -> more people classified as 'survived' (more false positives)")
print("Higher threshold -> fewer people classified as 'survived' (more false negatives)")
print("The model didn't change. Only the cutoff for calling something class 1 changed.")

---
## Step 6: The Real-World Production Problem — The Accuracy Paradox

**The scenario:** A fraud detection model reports 94.5% accuracy. The team is happy. The model ships.

**What's actually happening:** only 5% of transactions are fraud. A model that predicts "not fraud" for EVERY transaction would also score 94.5-95% accuracy — without ever catching a single fraud case.

This is the **accuracy paradox**: on imbalanced data, accuracy is a misleading metric. High accuracy can hide a model that's completely useless for the thing it was built to do.

Let's reproduce this exactly.

In [ ]:
np.random.seed(42)
n_fraud = 1000

transaction_amount = np.random.exponential(100, n_fraud)
is_fraud = (np.random.rand(n_fraud) < 0.05).astype(int)   # 5% fraud rate
transaction_amount[is_fraud == 1] += np.random.exponential(200, is_fraud.sum())

X_fraud = transaction_amount.reshape(-1, 1)
y_fraud = is_fraud

Xf_train, Xf_test, yf_train, yf_test = train_test_split(
    X_fraud, y_fraud, test_size=0.2, random_state=42, stratify=y_fraud
)

fraud_model = LogisticRegression(max_iter=1000).fit(Xf_train, yf_train)
fraud_pred = fraud_model.predict(Xf_test)
fraud_proba = fraud_model.predict_proba(Xf_test)[:, 1]

print(f"Overall fraud rate: {y_fraud.mean()*100:.1f}%")
print(f"Model accuracy: {accuracy_score(yf_test, fraud_pred)*100:.1f}%")
print()

cm_fraud = confusion_matrix(yf_test, fraud_pred)
print("Confusion Matrix:")
print(f"                  Predicted: Not Fraud   Predicted: Fraud")
print(f"Actual: Not Fraud       {cm_fraud[0,0]:>4}                {cm_fraud[0,1]:>4}")
print(f"Actual: Fraud           {cm_fraud[1,0]:>4}                {cm_fraud[1,1]:>4}")
print()

baseline_pred = np.zeros_like(yf_test)
baseline_acc = accuracy_score(yf_test, baseline_pred)
print(f"Baseline (predict 'not fraud' for everything) accuracy: {baseline_acc*100:.1f}%")
print()
print("The model's accuracy is barely better than always saying 'not fraud'.")
print("It caught", cm_fraud[1,1], "out of", cm_fraud[1,0]+cm_fraud[1,1], "actual fraud cases.")

In [ ]:
print("=== Fix: Lower the threshold to catch more fraud ===")
print(f"{'Threshold':>10} | {'Fraud Caught':>13} | {'False Alarms':>13} | {'Accuracy':>9}")
print("-" * 54)

total_fraud = yf_test.sum()

for thresh in [0.5, 0.3, 0.1, 0.05]:
    preds_t = (fraud_proba >= thresh).astype(int)
    cm_t = confusion_matrix(yf_test, preds_t)
    caught = cm_t[1,1]
    false_alarms = cm_t[0,1]
    acc_t = accuracy_score(yf_test, preds_t)
    print(f"{thresh:>10.2f} | {caught:>4}/{total_fraud:<8} | {false_alarms:>13} | {acc_t*100:>8.1f}%")

print()
print("At threshold=0.05, the model catches more fraud but accuracy DROPS.")
print("This is the tradeoff: in fraud detection, missing fraud (false negative)")
print("is usually far more costly than a false alarm (false positive).")
print()
print("Lesson: for imbalanced problems, never report accuracy alone.")
print("Use precision, recall, and a threshold chosen based on business cost,")
print("not the default 0.5.")

---
## Step 7: Summary — Logistic Regression Rules

In [ ]:
print("=" * 62)
print("DAY 14 SUMMARY: Logistic Regression")
print("=" * 62)
print()
print("WHY LOGISTIC, NOT LINEAR")
print("-" * 50)
print("1. Linear regression can output any number — not a valid probability")
print("2. Sigmoid squashes w·x+b into (0,1) — always a valid probability")
print("3. Logistic regression is far less sensitive to outliers in x")
print()
print("DECISION BOUNDARY")
print("-" * 50)
print("4. sigmoid(z)=0.5 when z=0, i.e. when w·x+b=0 -> the decision boundary")
print("5. Default threshold is 0.5, but it's a CHOICE, not a law")
print("6. Lowering the threshold catches more positives, at the cost of")
print("   more false alarms")
print()
print("EVALUATION")
print("-" * 50)
print("7. On imbalanced data, accuracy is misleading (the accuracy paradox)")
print("8. A model can score 94%+ accuracy while catching 0 positive cases")
print("9. Always check the confusion matrix, precision, recall — not just accuracy")
print("10. Choose the threshold based on the real cost of false negatives")
print("    vs false positives for YOUR problem")
print()
print("=" * 62)

---
## Practice Exercise

A loan default dataset is given below.

Your tasks:
1. Train a `LogisticRegression` model to predict `defaulted`
2. Print accuracy, ROC-AUC, and the confusion matrix
3. Check the default rate — is this dataset imbalanced?
4. Try thresholds 0.5, 0.3, and 0.2. How does the confusion matrix change?
5. If a missed default costs the bank 10x more than a false alarm, which threshold would you recommend, and why?

In [ ]:
# Practice dataset — loan default prediction
np.random.seed(21)
n_practice = 600

credit_score = np.random.randint(300, 850, n_practice).astype(float)
debt_ratio   = np.random.uniform(0, 1, n_practice)
income       = np.random.randint(20000, 150000, n_practice).astype(float)

logit_p = -4 + (-0.01)*(credit_score - 600) + 5*debt_ratio + (-0.00002)*(income - 50000)
default_prob = sigmoid(logit_p)
defaulted = (np.random.rand(n_practice) < default_prob).astype(int)

practice_df = pd.DataFrame({
    'credit_score': credit_score,
    'debt_ratio': debt_ratio,
    'income': income,
    'defaulted': defaulted
})

print("Practice dataset (loan defaults):")
print(practice_df.head(8).round(3).to_string(index=False))
print(f"\nShape: {practice_df.shape}")
print(f"Default rate: {defaulted.mean()*100:.1f}%")
print()
print("Your tasks:")
print("  1. Train LogisticRegression")
print("  2. Print accuracy, ROC-AUC, confusion matrix")
print("  3. Is this dataset imbalanced?")
print("  4. Try thresholds 0.5, 0.3, 0.2")
print("  5. Recommend a threshold given a 10x cost asymmetry")

# --- Your solution below ---


In [ ]:
# SOLUTION — try on your own first before looking here

X_loan = practice_df[['credit_score','debt_ratio','income']].values
y_loan = practice_df['defaulted'].values

Xtr, Xte, ytr, yte = train_test_split(X_loan, y_loan, test_size=0.2, random_state=42, stratify=y_loan)

loan_model = LogisticRegression(max_iter=1000).fit(Xtr, ytr)
loan_pred  = loan_model.predict(Xte)
loan_proba = loan_model.predict_proba(Xte)[:,1]

print(f"Accuracy: {accuracy_score(yte, loan_pred):.4f}")
print(f"ROC-AUC:  {roc_auc_score(yte, loan_proba):.4f}")
print(f"Default rate: {y_loan.mean()*100:.1f}%")
print("\nConfusion Matrix (threshold=0.5):")
print(confusion_matrix(yte, loan_pred))

print("\n=== Threshold comparison ===")
for t in [0.5, 0.3, 0.2]:
    p = (loan_proba >= t).astype(int)
    cm = confusion_matrix(yte, p)
    fn = cm[1,0]  # missed defaults
    fp = cm[0,1]  # false alarms
    print(f"  threshold={t}: missed defaults={fn}, false alarms={fp}, accuracy={accuracy_score(yte,p)*100:.1f}%")

print("\nRecommendation:")
print("If a missed default costs 10x a false alarm, accept more false alarms")
print("to reduce missed defaults. Lower the threshold from 0.5 toward 0.2-0.3,")
print("as long as the false alarm volume stays operationally manageable.")

---
## What's Next

**Day 15: Decision Trees**  
Gini impurity, entropy, and how splits are chosen. Why insurance companies use decision trees because regulators demand explainability.

---
**GitHub repo:** https://github.com/VaishnaviJagtap18/42-days-aiml-challenge  
**LinkedIn:** Follow for Day 15 tomorrow  
#42DaysOfML #MachineLearning #Python #MLEngineer